In [1]:
import os
import pandas as pd
from PIL import Image
from tqdm import tqdm

In [2]:
train_df = pd.read_csv("./data/processed/train.csv")
val_df = pd.read_csv("./data/processed/validation.csv")
test_df = pd.read_csv("./data/processed/test.csv")

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 31314
Validation: 3914
Test: 3915


In [3]:
def is_valid_image(image_path):
    """
    Returns True if image exists and can be opened.
    """

    if not os.path.exists(image_path):
        return False

    try:
        with Image.open(image_path) as img:
            img.verify()

        return True

    except Exception:
        return False

In [4]:
print(is_valid_image("./images/cmeazmz.jpg"))

False


In [5]:
print(is_valid_image("./images/100n2e.jpg"))

True


In [6]:
bad_images = []
good_images = []

image_files = os.listdir("./images")

for image_name in tqdm(image_files):

    image_path = os.path.join("./images", image_name)

    if is_valid_image(image_path):
        good_images.append(image_name)
    else:
        bad_images.append(image_name)

print(f"Good Images : {len(good_images)}")
print(f"Bad Images  : {len(bad_images)}")

100%|██████████| 39143/39143 [03:34<00:00, 182.32it/s] 

Good Images : 39113
Bad Images  : 30


In [7]:
bad_df = pd.DataFrame({"filename": bad_images})

bad_df.to_csv("bad_images.csv", index=False)

print("Saved bad_images.csv")

Saved bad_images.csv


In [8]:
bad_ids = set()

for file in bad_images:
    bad_ids.add(os.path.splitext(file)[0])

print("Bad IDs:", len(bad_ids))

Bad IDs: 30


In [9]:
def remove_bad_images(df):

    before = len(df)

    df = df[~df["id"].isin(bad_ids)].copy()

    after = len(df)

    print(f"Removed {before-after} rows")

    return df

In [10]:
train_clean = remove_bad_images(train_df)
val_clean = remove_bad_images(val_df)
test_clean = remove_bad_images(test_df)

Removed 18 rows
Removed 5 rows
Removed 7 rows


In [11]:
train_clean.to_csv("./data/processed/train_clean.csv", index=False)
val_clean.to_csv("./data/processed/validation_clean.csv", index=False)
test_clean.to_csv("./data/processed/test_clean.csv", index=False)

print("Clean CSVs saved.")

Clean CSVs saved.


In [12]:
from tqdm import tqdm
import os

def verify_dataset(df, images_dir):

    invalid_rows = []

    for idx, row in tqdm(df.iterrows(), total=len(df)):

        image_path = os.path.join(images_dir, row["id"] + ".jpg")

        if not is_valid_image(image_path):
            invalid_rows.append(row["id"])

    return invalid_rows

In [13]:
train_invalid = verify_dataset(train_clean, "./images")
val_invalid = verify_dataset(val_clean, "./images")
test_invalid = verify_dataset(test_clean, "./images")

print("Train Invalid:", len(train_invalid))
print("Validation Invalid:", len(val_invalid))
print("Test Invalid:", len(test_invalid))

100%|██████████| 3908/3908 [00:00<00:00, 4088.91it/s]

Train Invalid: 0
Validation Invalid: 0
Test Invalid: 0


In [15]:
print(train_clean["2_way_label"].value_counts())

2_way_label
1    17392
0    13904
Name: count, dtype: int64


In [16]:
print(len(train_df))
print(len(train_clean))

31314
31296
